In [ ]:
%load_ext autoreload
%autoreload 2

import logging
logging.basicConfig(level=logging.INFO)
from rexpand_pyutils_file import write_file

from utils.conversation_data import load_conversation_data
from utils.json import to_json_compatible
from models.context import Context
from models.llm_result import FulfilledAction
from nodes.orchestrator import orchestrate
from models.workflow import State


In [ ]:
# Load all conversations
conversations = load_conversation_data('./input/convo_2454_rows.xlsx')


In [ ]:
# Select a conversation to process
index = 6
messages_end = 5

# In this case, the referrer asks for some actions to be performed.
messages = conversations[index][:messages_end]

context = Context(messages=messages)
write_file('./output/current_messages.json', to_json_compatible(messages))

state = State(context=context)
write_file('./output/state.json', { "state": to_json_compatible(state.model_dump())})


In [ ]:
# Fresh start
state = orchestrate(state)

# The first orchestration pass should classify the conversation, summarize
# the required actions, and pause before generating a reply.

print(state.step)
print(state.classified_category)
print(state.required_actions)
print(state.suggested_topics)
print(state.selected_topics)
print(state.generated_reply_message)


In [ ]:
# Simulate the job seeker fulfilling the required actions. In a real app,
# these values would come from UI form inputs or uploaded attachment text.
state.fulfilled_actions = [
    FulfilledAction(
        action_id=action.action_id,
        answer=(
            "I can provide my resume, email, phone number, and the target "
            "job link requested by the referrer."
        ),
    )
    for action in state.required_actions.actions
]

# Re-run the orchestration process with fulfilled actions to generate 
#  the reply message
state = orchestrate(state)

print(state.step)
print(state.classified_category)
print(state.required_actions)
print(state.fulfilled_actions)
print(state.inferred_result)
print(state.suggested_topics)
print(state.selected_topics)
print(state.generated_reply_message)


In [ ]:
print(state.generated_reply_message.message)
